<a href="https://colab.research.google.com/github/jinbumlee95/Python_CLI_Board_Practice/blob/main/Python_CLI_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import clear_output
import pandas as pd
import traceback
from datetime import date, datetime
import difflib


running = True
recommend = None
session_id = None

# pandas 를 임시 데이터 베이스로 사용하도록 처리.
try:
    article_list = pd.read_csv("article_list.csv").convert_dtypes()
except FileNotFoundError:
    article_list = pd.DataFrame(columns=["No", "title", "desc","createdby","createdon","updatedby","lastupdated"])
    article_list.to_csv('article_list.csv',index=False)
finally :
    article_list['createdby'] = article_list['createdby'].astype('str')
    article_list['createdon'] = article_list['createdon'].astype('str')
    article_list['lastupdated'] = article_list['lastupdated'].astype('str')
    article_list['updatedby'] = article_list['updatedby'].astype('str')

try:
    member_list = pd.read_csv("member_list.csv",dtype="str").convert_dtypes()
except FileNotFoundError:
    member_list = pd.DataFrame(columns=["memberid", "password", "membername"])
    member_list.to_csv('member_list.csv',index=False)
finally :
    member_list['memberid'] = member_list['memberid'].astype('str')


def write_article() :
    """
    게시글을 작성하는 명령어 입니다. article write
    """
    global article_list
    global session_id
    # 1. 새로운 글 번호 계산 (기존 데이터가 있으면 max + 1, 없으면 1)
    if article_list.empty:
        new_no = 1
    else:
        new_no = article_list['No'].max() + 1

    title = input("제목 : ")
    print()
    desc = input("내용 : " )
    print()
    article_list.loc[len(article_list)] = [new_no, title, desc,session_id,datetime.now().strftime("%Y-%m-%d %H:%M:%S"),None,None]
    article_list.to_csv('article_list.csv', index=False) # 파일에 저장
    print("{}번 글이 생성되었습니다.".format(new_no))
    read_article(new_no)

def list_article(search = None):
    """
    게시글 목록을 조회하는 명령어 입니다. article list , article list searchtext
    """
    global article_list
    global member_list
    if article_list.empty :
        print('게시글이 존재하지 않습니다.')
    else :
        # 작성자 명을 가져오기 위해서 merge
        list_table = pd.merge(
            article_list,
            member_list,
            left_on='createdby',
            right_on='memberid',
            how='left'
        )
        #.str.contains('apple')
        if search :
            list_table = list_table[list_table['title'].str.contains(search)]
        print('-'*65)
        print(f"{'No':<5} | {'title':<30} | {'createdby':<10} | {'createdon':<20}")
        print('-'*65)
        for idx, board in list_table.loc[::-1].iterrows() :
            #print(board)
            day = board['createdon']
            today = str(date.today())
            if(day.startswith(today)) :
                day = day.replace(today,'').strip()[0:5]
            else :
                day = day[0:10]
            print(f"{board['No']:<5} | {board['title']:<30} | {board['membername']:<10} | {day:<20}")
            print('-'*65)

def read_article(n = None) :
    """
    게시글을 조회하는 명령어 입니다. article read number
    """
    n = check_number_input(n,'조회')

    board = article_list[article_list['No'] == n]

    if board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    else :
        board = board.iloc[0]
        print()
        print("------------------------------------------")
        print(f"TITLE      | {board['title']}")
        print("------------------------------------------")
        print(f"CREATEDON  | {board['createdon']}")
        print("------------------------------------------")
        print(f"DESC       | {board['desc']}")
        print("------------------------------------------")
        print()


def delete_article(n = None):
    """
    게시글을 삭제하는 명령어 입니다. article delete number
    """
    global article_list

    n = check_number_input(n,'삭제')

    board = article_list[article_list['No'] == n]
    if board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    elif board[board['createdby'] == session_id].empty :
        print("==본인의 게시글만 삭제 할 수 있습니다.==")
    else :
        # 3. 해당 번호를 제외한 행만 남겨 삭제 처리
        article_list = article_list[article_list['No'] != n].copy()
        # 4. 'No' 컬럼 번호 1부터 재정렬 > 이거는 게시글 순번이 필요하니까 안할거임
        #article_list['No'] = range(1, len(article_list) + 1)
        # 5. CSV 파일 저장
        article_list.to_csv('article_list.csv', index=False)

        print('삭제되었습니다.')

def close_cli() :
    """
    CLI 를 종료하는 명령어 입니다.
    """
    global running
    running = False
    print("=======  CLI 게시판 종료 =========")

def print_help(*args):
    """
    명령어를 안내해줍니다. help article list
    """
    if not args :
        print("COMMANDS ")
        print('========================================')
        for key in command_dict :
            print(key)
        print('========================================')
        print('if you need more information , type help somthing (ex) help login')
    else :
        #여기서 함수의
        function = ' '.join(args)
        if function in command_dict :
            print(command_dict[function].__doc__)
        else :
            print('해당 명령어는 존재 하지 않습니다.')

def update_article(n = None):
    """
    게시글을 수정하는 명령어 입니다. article update number
    """
    global session_id
    n = check_number_input(n,'수정')

    update_board = article_list[article_list['No'] == n]

    if update_board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    elif update_board[update_board['createdby'] == session_id].empty :
        print("==본인의 게시글만 수정 할 수 있습니다.==")
    else :
        update_board = article_list[article_list['No'] == n].iloc[0]
        print('-----------------------------------------------')
        print('-----------------UPDATE BEFORE-----------------')
        print('-----------------------------------------------')
        read_article(n)

        print('수정할 사항을 선택 해 주세요.')
        print('1 : 제목')
        print('2 : 내용')

        # 1이나 2가 입력될 때까지 계속 입력 받기
        mode = input("선택 ) ")
        while mode not in ['1', '2']:
            print('올바른 번호를 선택해 주세요. (1 : 제목, 2 : 내용 , 3 : 나가기)')
            mode = input("선택 ) ")

        if mode == '1':
            title = input('수정할 제목을 입력 해 주세요 : ').strip()
            if title:
                article_list.loc[article_list['No'] == n, 'title'] = title
                article_list.loc[article_list['No'] == n, 'lastupdated'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                article_list.loc[article_list['No'] == n, 'updatedby'] = session_id

        elif mode == '2':
            desc = input('수정할 내용을 입력 해 주세요 : ').strip()
            if desc:
                article_list.loc[article_list['No'] == n, 'desc'] = desc
                article_list.loc[article_list['No'] == n, 'lastupdated'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                article_list.loc[article_list['No'] == n, 'updatedby'] = session_id
        elif mode == '3' :
            pass
        article_list.to_csv('article_list.csv', index=False) # 파일에 저장
        print('-----------------------------------------------')
        print('-----------------UPDATE AFTER------------------')
        print('-----------------------------------------------')
        read_article(int(n)) # 수정 후에 수정 한 게시글을 조회 하도록 수정


def check_number_input(n, text) :
    if n == None :
        n = input(text+"할 게시글 번호 : ")
        while not n.isdigit() :
            n = input(text+"할 게시글 번호 : ")
        n = int(n)
    else :
        if not str(n).isdigit() :
            n = input(text+"할 게시글 번호 : ")
            while not n.isdigit() :
                n = input(text+"할 게시글 번호 : ")

        n = int(n)

    return n





def login_process(id = None , password = None):
    """
    로그인 명령어 입니다 login id password
    """
    global session_id
    global membername

    if not id :
        id = input('아이디를 입력해주세요 : ')
    if not password :
        password = input('패스워드를 입력해주세요 : ')

    if member_list[(member_list['memberid'] == id) & (member_list['password'] == password)].empty:
        print('존재하지 않는 회원입니다')
    else :
        member = member_list[(member_list['memberid'] == id)].iloc[0]
        session_id = id
        membername = member['membername']
        print(f'{membername}님, 방문을 환영합니다.')


def logout_process() :
    """
    로그아웃 명령어 입니다. logout
    """
    global session_id
    global membername

    print(f'{membername}님, 재방문을 기다리겠습니다.')

    session_id = None
    membername = None

def sign_up():
    """
    회원가입 명령어 입니다.
    """
    global member_list

    memberid = input("아이디를 입력해 주세요 : ")
    while not memberid :
        memberid = input("아이디를 입력해 주세요 : ")

    duplication_check = (member_list['memberid'] == memberid).any()
    while duplication_check :
        duplication_check = (member_list['memberid'] == memberid).any()
        if(duplication_check) :
            memberid = input("중복된 아이디 입니다. 다른 아이디를 입력해주세요. : ")
    print()
    password = input("비밀번호를 입력해 주세요 : " )
    while not  password :
        password = input("비밀번호를 입력해 주세요 : " )
    print()
    membername = input("이름을 입력해 주세요 : " )
    while not  membername :
        membername = input("이름을 입력해 주세요 : " )
    print()
    member_list.loc[len(member_list)] = [memberid, password, membername]
    member_list.to_csv('member_list.csv', index=False) # 파일에 저장
    print("{}님 환영합니다. 로그인 후 사용해주세요.".format(membername))



def clear_outputs() :
    """
    CLI 화면출력을 전부 지웁니다.
    """
    clear_output()

def do_recommend() :
    """
    오탈자 등으로 인한 추천명령어가 있을경우, 해당 명령어를 실행합니다.
    """
    global recommend
    global session_id
    if recommend :
        if recommend.startswith('article') and  session_id == None:
            print('로그인 후 사용해주세요')
        else :
            command_dict[recommend]()
        recommend = None
    else :
        print('추천 명령어가 없습니다.')


command_dict = {

    "article write" : write_article,
    "article list" : list_article,
    "article read" : read_article,
    "article delete" : delete_article,
    "article update" : update_article,
    "help" : print_help,
    "clear" : clear_outputs,
    'login' : login_process,
    'logout' : logout_process,
    'do' : do_recommend,
    'sign' : sign_up,
    "exit" : close_cli
}



print("=======  CLI 게시판 실행 =========")
while running :

    command = input("명령어 ) ").strip()

    if not command :
        continue

    command_list = command.split()

    # 2단어 명령어(article read 등)와 1단어 명령어(help, exit 등) 구분 처리
    if len(command_list) >= 2 and f"{command_list[0]} {command_list[1]}" in command_dict:
        # 2단 커맨드가 dict 안에 있으면 cmd 를 2단 커맨드로

        cmd = f"{command_list[0]} {command_list[1]}"
        args = command_list[2:]
    else:
        # 아니면 1단 커맨드에 뒤에 나오는건 인자들
        cmd = command_list[0]
        args = command_list[1:]





    if cmd not in command_dict :
        possible_matches = difflib.get_close_matches(command, command_dict.keys(), n=1, cutoff=0.4)
        if possible_matches and possible_matches[0] != 'do':
            recommend = possible_matches[0]
            print('-----------------------------------------------')
            print(f'{command}는 존재하지 않는 명령어 입니다.\n')
            print(f"혹시 {possible_matches[0]}를 찾으셨나요?\n")
            print(f"해당 명령어를 실행 하시려면 do 를 입력해주세요.\n")
            print("아니라면, help 명령어를 통해, 명령어 목록을 확인해주세요.")
            print('-----------------------------------------------')
        else :
            print('-----------------------------------------------')
            print(f'{command}는 존재하지 않는 명령어 입니다.\n')
            print("help 명령어를 통해, 명령어 목록을 확인해주세요.")
            print('-----------------------------------------------')
    else :

        if cmd not in  ['login','sign','help','exit','clear','do'] and session_id == None :
            print('로그인 후 사용 해주세요.')
            continue
        elif cmd in  ['login','sign'] and session_id != None:
            print('로그아웃 후 사용해주세요.')
            continue
        elif cmd == 'logout' and session_id == None :
            print('로그인 상태가 아닙니다.')
            continue

        try :
            if args :
                command_dict[cmd](*args)
            else :
                command_dict[cmd]()
        except:
            # 인자 수를 잘못 넘겨주는 경우에 대해서 예외 처리
            print('잘못된 명령어를 입력하셨습니다.')
            #exc_str = traceback.format_exc()
            #print(exc_str)



=======  CLI 게시판 실행 =========
명령어 ) login
아이디를 입력해주세요 : member1
패스워드를 입력해주세요 : 1234
JB님, 방문을 환영합니다.
명령어 ) alist
-----------------------------------------------
alist는 존재하지 않는 명령어 입니다.

혹시 article list를 찾으셨나요?

해당 명령어를 실행 하시려면 do 를 입력해주세요.

아니라면, help 명령어를 통해, 명령어 목록을 확인해주세요.
-----------------------------------------------
명령어 ) do
-----------------------------------------------------------------
No    | title                          | createdby  | createdon           
-----------------------------------------------------------------
7     | 4번글                            | JB         | 03:31               
-----------------------------------------------------------------
6     | 444444                         | JB         | 03:29               
-----------------------------------------------------------------
5     | 4번 글이에요                        | JB         | 03:28               
-----------------------------------------------------------------
4     | 4번 글이에요                      